# Importing modules and functions

In [128]:
import numpy as np
import pandas as pd
from rdkit import Chem, DataStructs
from rdkit.Chem import AllChem, Descriptors
from rdkit.Chem import MACCSkeys
import chembl_structure_pipeline
from molvs import standardize_smiles
from sklearn.model_selection import KFold, GridSearchCV
from sklearn.model_selection import permutation_test_score
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_predict
from sklearn import metrics
from sklearn.metrics import pairwise_distances
import joblib
import pickle
from numpy import savetxt
from padelpy import from_sdf
from IPython.display import HTML
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score
from catboost import CatBoostRegressor
from sklearn.neighbors import KNeighborsRegressor
import warnings
warnings.filterwarnings('ignore')

# Data entry and curation work set

In [2]:
uploaded_file_ws="datasets/HDAC8_work.sdf"
supplier_ws = Chem.ForwardSDMolSupplier(uploaded_file_ws,sanitize=False)
failed_mols_ws = []
all_mols_ws =[]
wrong_structure_ws=[]
wrong_smiles_ws=[]
y_tr = []
y_bad_index=[]

for i, m in enumerate(supplier_ws):
    structure = Chem.Mol(m)
    all_mols_ws.append(structure)
    y_tr.append(m.GetProp("pchembl_value_mean"))
    try:
        Chem.SanitizeMol(structure)
    except:
        failed_mols_ws.append(m)
        wrong_smiles_ws.append(Chem.MolToSmiles(m))
        wrong_structure_ws.append(str(i+1))
        y_bad_index.append(i)
print('Original data: ', len(all_mols_ws), 'molecules')
print('Failed data: ', len(failed_mols_ws), 'molecules')
number_ws =[]
for i in range(len(failed_mols_ws)):
        number_ws.append(str(i+1))
bad_molecules_ws = pd.DataFrame({'No. failed molecule in original set': wrong_structure_ws, 'SMILES of wrong structure: ': wrong_smiles_ws, 'No.': number_ws}, index=None)
bad_molecules_ws = bad_molecules_ws.set_index('No.')
bad_molecules_ws

Original data:  2210 molecules
Failed data:  0 molecules


,No. failed molecule in original set,SMILES of wrong structure:
No.,,


deleting activity values for substances with incorrect structure

In [3]:
y_tr[:] = [x for i,x in enumerate(y_tr) if i not in y_bad_index]

In [4]:
len(y_tr)

2210

# Standardization SDF file for work set

In [5]:
all_mols_ws[:] = [x for i,x in enumerate(all_mols_ws) if i not in y_bad_index] 
records = []
for i in range(len(all_mols_ws)):
    record = Chem.MolToSmiles(all_mols_ws[i])
    records.append(record)

moldf_ws = []
for i,record in enumerate(records):
    standard_record = standardize_smiles(record)
    m = Chem.MolFromSmiles(standard_record)
    moldf_ws.append(m)
    
print('Kept data: ', len(moldf_ws), 'molecules')

Kept data:  2210 molecules


In [6]:
moldf_ws=pd.DataFrame(moldf_ws, columns=['Mol'])
moldf_ws

,Mol
0,<rdkit.Chem.rdchem.Mol object at 0x00000220658...
1,<rdkit.Chem.rdchem.Mol object at 0x00000220647...
2,<rdkit.Chem.rdchem.Mol object at 0x00000220658...
3,<rdkit.Chem.rdchem.Mol object at 0x00000220658...
4,<rdkit.Chem.rdchem.Mol object at 0x00000220658...
...,...
2205,<rdkit.Chem.rdchem.Mol object at 0x00000220659...
2206,<rdkit.Chem.rdchem.Mol object at 0x00000220659...
2207,<rdkit.Chem.rdchem.Mol object at 0x00000220659...
2208,<rdkit.Chem.rdchem.Mol object at 0x00000220659...


# Data entry and curation test set

In [7]:
uploaded_file_ts="datasets/HDAC8_test.sdf"
supplier_ts = Chem.ForwardSDMolSupplier(uploaded_file_ts,sanitize=False)
failed_mols_ts = []
all_mols_ts =[]
wrong_structure_ts=[]
wrong_smiles_ts=[]
y_ts = []
y_bad_index=[]
for i, m in enumerate(supplier_ts):
    structure = Chem.Mol(m)
    all_mols_ts.append(structure)
    y_ts.append(m.GetProp("pchembl_value_mean"))
    try:
        Chem.SanitizeMol(structure)
    except:
        failed_mols_ts.append(m)
        wrong_smiles_ts.append(Chem.MolToSmiles(m))
        wrong_structure_ts.append(str(i+1))
        y_bad_index.append(i)
print('Original data: ', len(all_mols_ts), 'molecules')
print('Failed data: ', len(failed_mols_ts), 'molecules')
number_ts =[]
for i in range(len(failed_mols_ts)):
        number_ts.append(str(i+1))
bad_molecules_ts = pd.DataFrame({'No. failed molecule in original set': wrong_structure_ts, 'SMILES of wrong structure: ': wrong_smiles_ts, 'No.': number_ts}, index=None)
bad_molecules_ts = bad_molecules_ts.set_index('No.')
bad_molecules_ts

Original data:  553 molecules
Failed data:  0 molecules


,No. failed molecule in original set,SMILES of wrong structure:
No.,,


deleting activity values for substances with incorrect structure

In [8]:
y_ts[:] = [x for i,x in enumerate(y_ts) if i not in y_bad_index]

In [9]:
len(y_ts)

553

# Standardization SDF file for test set

In [10]:
all_mols_ts[:] = [x for i,x in enumerate(all_mols_ts) if i not in y_bad_index] 
records = []
for i in range(len(all_mols_ts)):
    record = Chem.MolToSmiles(all_mols_ts[i])
    records.append(record)

moldf_ts = []
for i,record in enumerate(records):
    standard_record = standardize_smiles(record)
    m = Chem.MolFromSmiles(standard_record)
    moldf_ts.append(m)
    
print('Kept data: ', len(moldf_ts), 'molecules')

Kept data:  553 molecules


In [11]:
moldf_ts=pd.DataFrame(moldf_ts, columns=['Mol'])
moldf_ts

,Mol
0,<rdkit.Chem.rdchem.Mol object at 0x00000220658...
1,<rdkit.Chem.rdchem.Mol object at 0x00000220658...
2,<rdkit.Chem.rdchem.Mol object at 0x00000220658...
3,<rdkit.Chem.rdchem.Mol object at 0x00000220658...
4,<rdkit.Chem.rdchem.Mol object at 0x00000220658...
...,...
548,<rdkit.Chem.rdchem.Mol object at 0x00000220658...
549,<rdkit.Chem.rdchem.Mol object at 0x00000220658...
550,<rdkit.Chem.rdchem.Mol object at 0x00000220658...
551,<rdkit.Chem.rdchem.Mol object at 0x00000220658...


# Calculation MACCS Fingerprint for work set

In [12]:
def calcfp(mol):
    fp = MACCSkeys.GenMACCSKeys(mol)
    fp = pd.Series(np.asarray(fp))
    fp = fp.add_prefix('MACCS_')
    return fp

# Training set
desc_ws = moldf_ws.Mol.apply(calcfp)
desc_ws

,MACCS_0,MACCS_1,MACCS_2,MACCS_3,MACCS_4,MACCS_5,MACCS_6,MACCS_7,MACCS_8,MACCS_9,...,MACCS_157,MACCS_158,MACCS_159,MACCS_160,MACCS_161,MACCS_162,MACCS_163,MACCS_164,MACCS_165,MACCS_166
0,0,0,0,0,0,0,0,0,0,0,...,1,1,1,1,1,1,1,1,1,0
1,0,0,0,0,0,0,0,0,0,0,...,1,0,1,0,0,1,1,1,1,0
2,0,0,0,0,0,0,0,0,0,0,...,1,1,1,0,1,1,1,1,1,0
3,0,0,0,0,0,0,0,0,0,0,...,1,1,1,0,1,1,1,1,1,0
4,0,0,0,0,0,0,0,0,0,0,...,1,1,1,0,1,1,1,1,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2205,0,0,0,0,0,0,0,0,0,0,...,1,1,1,0,1,1,1,1,1,0
2206,0,0,0,0,0,0,0,0,0,0,...,0,1,1,0,1,1,1,1,1,0
2207,0,0,0,0,0,0,0,0,0,0,...,0,1,1,0,1,1,1,1,1,0
2208,0,0,0,0,0,0,0,0,0,0,...,0,1,1,0,1,1,1,1,1,0


In [13]:
y_tr = np.array(y_tr, dtype=np.float32)
len(y_tr)

2210

# Calculation MACCS Fingerprint for test set

In [14]:
desc_ts = moldf_ts.Mol.apply(calcfp)
desc_ts

,MACCS_0,MACCS_1,MACCS_2,MACCS_3,MACCS_4,MACCS_5,MACCS_6,MACCS_7,MACCS_8,MACCS_9,...,MACCS_157,MACCS_158,MACCS_159,MACCS_160,MACCS_161,MACCS_162,MACCS_163,MACCS_164,MACCS_165,MACCS_166
0,0,0,0,0,0,0,0,0,0,0,...,0,1,1,0,1,1,1,1,1,0
1,0,0,0,0,0,0,0,0,0,0,...,0,1,1,1,1,1,1,1,1,0
2,0,0,0,0,0,0,0,0,0,0,...,0,1,1,0,1,1,1,1,1,0
3,0,0,0,0,0,0,0,0,0,0,...,0,1,0,0,1,1,1,1,1,0
4,0,0,0,0,0,0,0,0,0,0,...,1,1,1,0,1,1,1,1,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
548,0,0,0,0,0,0,0,0,0,0,...,0,1,1,0,1,1,1,1,1,0
549,0,0,0,0,0,0,0,0,0,0,...,1,1,1,1,1,1,1,1,1,0
550,0,0,0,0,0,0,0,0,0,0,...,1,1,1,1,1,1,1,1,1,0
551,0,0,0,0,0,0,0,0,0,0,...,0,1,1,0,1,1,1,1,1,0


# BASELINE

 ## GradientBoostingRegressor model building and validation

In [52]:
seed = 42

In [53]:
cv=KFold(n_splits=5, random_state=seed, shuffle=True)

In [23]:
%%time
model = CatBoostRegressor()
parameters = {'depth' : [6,8,10],
              'learning_rate' : [0.01, 0.05, 0.1],
              'iterations'    : [100,500, 1000]
              }

grid = GridSearchCV(estimator=model, param_grid = parameters, n_jobs=-1, cv = cv)
grid.fit(desc_ws, y_tr, verbose=False)

CPU times: total: 4min 13s
Wall time: 13min 17s


,estimator,CatBoostRegre...nction='RMSE')
,param_grid,"{'depth': [6, 8, ...], 'iterations': [100, 500, ...], 'learning_rate': [0.01, 0.05, ...]}"
,scoring,None
,n_jobs,-1
,refit,True
,cv,KFold(n_split... shuffle=True)
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False


In [24]:
best_CatBR = grid.best_estimator_

In [25]:
grid.best_params_

{'depth': 10, 'iterations': 500, 'learning_rate': 0.05}

In [16]:
y_pred_ws_GBR = best_CatBR.predict(desc_ws)

In [17]:
R2_WS = round(r2_score(y_tr, y_pred_ws_GBR), 2)
R2_WS

0.91

In [18]:
RMSE_WS=float(round(np.sqrt(mean_squared_error(y_tr, y_pred_ws_GBR)), 2))
RMSE_WS

0.25

In [26]:
params={'verbose': False}

In [27]:
%%time
y_pred_CV_CatBR = cross_val_predict(best_CatBR, desc_ws, y_tr, cv=cv, params=params)

CPU times: total: 19min 30s
Wall time: 1min 27s


In [28]:
Q2_CV = round(r2_score(y_tr, y_pred_CV_CatBR), 2)
Q2_CV

0.55

In [29]:
RMSE_CV = float(round(np.sqrt(mean_squared_error(y_tr, y_pred_CV_CatBR)), 2))
RMSE_CV

0.57

In [19]:
y_pred_GBR = best_CatBR.predict(desc_ts)

In [20]:
Q2_TS = round(r2_score(y_ts, y_pred_GBR), 2)
Q2_TS

0.57

In [21]:
RMSE_TS=float(round(np.sqrt(mean_squared_error(y_ts, y_pred_GBR)), 2))
RMSE_TS

0.56

save the model to disk

In [33]:
pickle.dump(best_CatBR, open('models/MACCS/CatBoost_MACCS.pkl', 'wb'))

load the model from disk

In [15]:
best_CatBR = pickle.load(open('models/MACCS/CatBoost_MACCS.pkl', 'rb'))

# Y-randomization

In [34]:
params={'verbose': False}

In [35]:
permutations = 50
score, permutation_scores, pvalue = permutation_test_score(best_CatBR, desc_ws, y_tr,
                                                           cv=cv, scoring='r2',
                                                           n_permutations=permutations,
                                                           n_jobs=-1,
                                                           params=params,
                                                           random_state=seed)
print('True score = ', score.round(2),
      '\nY-randomization = ', np.mean(permutation_scores).round(2),
      '\np-value = ', pvalue.round(4))

True score =  0.55 
Y-randomization =  -0.23 
p-value =  0.0196


# Estimating applicability domain. Method - Euclidian distances, K=1

In [22]:
neighbors_k= pairwise_distances(desc_ws, n_jobs=-1)
neighbors_k.sort(0)

In [23]:
df_tr=pd.DataFrame(neighbors_k)
df_tr

,0,1,2,3,4,5,6,7,8,9,...,2200,2201,2202,2203,2204,2205,2206,2207,2208,2209
0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,3.316625,2.645751,1.000000,0.000000,2.449490,1.732051,1.414214,0.000000,2.828427,2.645751,...,1.000000,3.162278,3.605551,1.414214,3.162278,4.000000,0.000000,0.000000,1.000000,2.000000
2,3.316625,4.358899,2.645751,2.000000,2.449490,2.828427,1.414214,2.000000,3.000000,2.645751,...,2.236068,3.162278,3.605551,2.000000,3.162278,4.123106,1.414214,1.414214,1.414214,2.000000
3,3.605551,4.358899,3.316625,2.000000,2.828427,4.000000,2.000000,2.000000,3.162278,3.162278,...,2.828427,3.162278,3.605551,2.000000,3.316625,4.123106,1.414214,1.414214,2.000000,2.449490
4,3.605551,4.358899,3.316625,2.000000,3.162278,4.000000,2.000000,2.000000,3.316625,4.000000,...,3.605551,3.162278,3.741657,2.236068,3.605551,4.123106,1.414214,1.414214,2.000000,2.449490
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2205,8.366600,8.602325,7.937254,8.124038,8.124038,8.774964,8.246211,8.124038,8.000000,7.874008,...,8.426150,8.124038,8.306624,8.124038,8.426150,8.062258,7.937254,7.937254,8.944272,7.874008
2206,8.366600,8.602325,8.000000,8.185353,8.124038,8.774964,8.246211,8.185353,8.000000,7.874008,...,8.485281,8.124038,8.306624,8.124038,8.426150,8.124038,7.937254,7.937254,9.000000,7.874008
2207,8.426150,8.602325,8.000000,8.185353,8.124038,8.774964,8.306624,8.185353,8.000000,7.874008,...,8.485281,8.185353,8.306624,8.185353,8.485281,8.124038,8.000000,8.000000,9.000000,7.874008
2208,8.485281,8.602325,8.000000,8.246211,8.124038,8.831761,8.306624,8.246211,8.000000,7.874008,...,8.544004,8.185353,8.485281,8.185353,8.485281,8.124038,8.000000,8.000000,9.055385,7.874008


In [24]:
similarity= neighbors_k

In [25]:
Dmean=np.mean(similarity[1,:])

In [26]:
round(Dmean, 2)

np.float64(1.46)

In [27]:
std=np.std(similarity[1,:])

In [28]:
round(std, 2)

np.float64(1.21)

In [29]:
model_AD_limit=Dmean+std*0.5
print(np.round(model_AD_limit, 2))

2.07


In [30]:
neighbors_k_ts= pairwise_distances(desc_ws,Y=desc_ts, n_jobs=-1)
neighbors_k_ts.sort(0)

In [31]:
x_ts_AD=pd.DataFrame(neighbors_k_ts)
x_ts_AD

,0,1,2,3,4,5,6,7,8,9,...,543,544,545,546,547,548,549,550,551,552
0,1.414214,3.316625,2.000000,3.162278,1.000000,1.414214,2.828427,1.000000,2.828427,1.414214,...,3.000000,2.000000,1.000000,1.732051,1.414214,1.732051,2.000000,0.000000,1.732051,0.000000
1,2.828427,3.605551,2.449490,3.316625,1.000000,1.414214,2.828427,1.000000,2.828427,1.732051,...,3.162278,2.645751,2.000000,1.732051,1.414214,2.645751,2.000000,0.000000,2.645751,0.000000
2,3.000000,3.741657,2.645751,3.605551,2.236068,2.000000,3.000000,2.000000,2.828427,2.000000,...,3.316625,3.000000,2.449490,1.732051,1.732051,2.645751,2.236068,1.000000,2.645751,2.000000
3,3.162278,3.741657,2.828427,3.741657,2.236068,2.449490,3.605551,2.236068,2.828427,2.000000,...,3.316625,3.000000,2.645751,2.236068,1.732051,2.645751,2.236068,1.000000,2.645751,2.000000
4,3.316625,3.741657,3.000000,3.741657,2.236068,2.449490,3.605551,2.236068,2.828427,2.236068,...,3.464102,3.162278,2.645751,2.236068,1.732051,2.645751,2.236068,1.000000,2.828427,2.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2205,7.874008,7.810250,8.000000,8.000000,8.062258,8.124038,8.062258,8.062258,7.937254,8.124038,...,7.810250,8.124038,7.937254,7.937254,8.426150,7.810250,8.426150,8.306624,8.124038,8.602325
2206,7.874008,7.810250,8.000000,8.062258,8.124038,8.185353,8.124038,8.062258,7.937254,8.185353,...,7.810250,8.124038,7.937254,7.937254,8.426150,7.810250,8.426150,8.306624,8.185353,8.602325
2207,7.937254,7.874008,8.000000,8.124038,8.124038,8.185353,8.185353,8.062258,7.937254,8.246211,...,7.937254,8.185353,8.000000,7.937254,8.426150,7.810250,8.485281,8.306624,8.185353,8.602325
2208,8.000000,7.874008,8.000000,8.124038,8.185353,8.246211,8.246211,8.062258,8.000000,8.306624,...,8.000000,8.185353,8.000000,7.937254,8.426150,7.810250,8.485281,8.306624,8.185353,8.602325


In [32]:
similarity_ts= neighbors_k_ts
cpd_AD=similarity_ts[0,:]
cpd_value = np.round(cpd_AD, 3)
print(cpd_value)

[1.414 3.317 2.    3.162 1.    1.414 2.828 1.    2.828 1.414 4.359 1.
 0.    3.464 3.464 1.414 1.414 1.    0.    3.317 0.    0.    1.414 1.
 1.414 1.414 2.236 1.414 0.    2.    0.    3.    1.414 2.449 1.    2.828
 1.414 2.    3.464 3.464 4.    3.    2.646 0.    2.    1.414 1.    2.
 0.    2.236 1.414 0.    2.449 2.449 1.    1.414 2.646 3.606 0.    0.
 0.    2.    1.732 1.732 1.732 2.828 1.414 1.732 2.449 1.414 2.236 1.
 0.    2.    2.236 0.    3.742 0.    2.449 0.    1.    0.    0.    2.236
 2.    2.646 1.732 0.    0.    1.732 1.414 1.732 1.414 1.414 2.    2.449
 3.464 0.    0.    2.    0.    1.    0.    2.    1.732 1.732 0.    2.449
 1.    0.    0.    2.236 2.    2.236 0.    1.    1.414 2.236 0.    1.732
 2.    0.    0.    1.732 0.    1.    0.    0.    1.    1.414 2.646 3.317
 1.732 3.    0.    2.    1.414 2.646 0.    3.162 1.    1.414 0.    1.414
 1.    1.    0.    2.449 1.    1.    0.    0.    0.    2.236 2.646 1.
 4.472 2.236 2.    2.236 2.449 1.414 1.414 1.414 1.    1.414 0.    2.

In [33]:
cpd_AD = np.where(cpd_value <= model_AD_limit, True, False)
print(cpd_AD)

[ True False  True False  True  True False  True False  True False  True
  True False False  True  True  True  True False  True  True  True  True
  True  True False  True  True  True  True False  True False  True False
  True  True False False False False False  True  True  True  True  True
  True False  True  True False False  True  True False False  True  True
  True  True  True  True  True False  True  True False  True False  True
  True  True False  True False  True False  True  True  True  True False
  True False  True  True  True  True  True  True  True  True  True False
 False  True  True  True  True  True  True  True  True  True  True False
  True  True  True False  True False  True  True  True False  True  True
  True  True  True  True  True  True  True  True  True  True False False
  True False  True  True  True False  True False  True  True  True  True
  True  True  True False  True  True  True  True  True False False  True
 False False  True False False  True  True  True  T

In [34]:
print("Coverage = ", round(sum(cpd_AD) / len(cpd_AD),2))

Coverage =  0.72


In [35]:
print("Indices of substances included in AD = ", np.where(cpd_AD != 0)[0])

Indices of substances included in AD =  [  0   2   4   5   7   9  11  12  15  16  17  18  20  21  22  23  24  25
  27  28  29  30  32  34  36  37  43  44  45  46  47  48  50  51  54  55
  58  59  60  61  62  63  64  66  67  69  71  72  73  75  77  79  80  81
  82  84  86  87  88  89  90  91  92  93  94  97  98  99 100 101 102 103
 104 105 106 108 109 110 112 114 115 116 118 119 120 121 122 123 124 125
 126 127 128 129 132 134 135 136 138 140 141 142 143 144 145 146 148 149
 150 151 152 155 158 161 162 163 164 165 166 168 169 170 171 172 173 174
 175 176 177 178 179 180 181 182 183 184 185 186 187 190 191 194 195 196
 199 200 202 205 206 207 208 209 211 213 214 217 218 219 220 221 222 224
 225 226 227 228 229 230 231 232 233 234 235 236 237 239 240 241 242 244
 245 246 247 248 249 250 251 252 256 258 259 261 262 263 264 265 266 267
 268 269 270 271 273 274 275 276 277 278 279 282 283 285 286 287 288 289
 290 293 294 295 296 298 299 301 304 305 308 310 311 312 313 315 316 319
 320 321 32

In [36]:
out_Ad=list(np.where(cpd_AD == 0)[0])

# Prediction only for molecules included in  AD

In [37]:
y_pred_GBR_ad=list(y_pred_GBR)

In [38]:
y_pred_GBR_ad[:] = [x for i,x in enumerate(y_pred_GBR_ad) if i not in out_Ad]

In [39]:
len(y_pred_GBR_ad)

399

In [40]:
y_ts_ad=list(y_ts)

In [41]:
y_ts_ad[:] = [x for i,x in enumerate(y_ts_ad) if i not in out_Ad]

In [42]:
len(y_ts_ad)

399

In [43]:
Q2_TS = round(r2_score(y_ts_ad, y_pred_GBR_ad), 2)
Q2_TS

0.66

In [44]:
RMSE_TS=float(round(np.sqrt(mean_squared_error(y_ts_ad, y_pred_GBR_ad)), 2))
RMSE_TS

0.5

# SVM model building and validation

In [66]:
from sklearn.svm import SVR

In [67]:
param_grid = {"C": [10 ** i for i in range(0, 5)],
              "gamma": [10 ** i for i in range(-6, 0)]}

In [68]:
seed = 42
cv=KFold(n_splits=5, random_state=seed, shuffle=True)

In [69]:
svm = GridSearchCV(SVR(C=1.0, epsilon=0.2), param_grid, n_jobs=-1, cv=cv, verbose=1)

In [70]:
svm.fit(desc_ws, y_tr)

Fitting 5 folds for each of 30 candidates, totalling 150 fits


,estimator,SVR(epsilon=0.2)
,param_grid,"{'C': [1, 10, ...], 'gamma': [1e-06, 1e-05, ...]}"
,scoring,None
,n_jobs,-1
,refit,True
,cv,KFold(n_split... shuffle=True)
,verbose,1
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,kernel,'rbf'


In [71]:
svm.best_params_
best_svm = svm.best_estimator_

In [72]:
svm.best_params_

{'C': 10, 'gamma': 0.1}

In [47]:
y_pred_ws_SVM = best_svm.predict(desc_ws)

In [48]:
R2_WS = round(r2_score(y_tr, y_pred_ws_SVM), 2)
R2_WS

0.92

In [49]:
RMSE_WS=float(round(np.sqrt(mean_squared_error(y_tr, y_pred_ws_SVM)), 2))
RMSE_WS

0.24

In [54]:
y_pred_CV_svm = cross_val_predict(best_svm, desc_ws, y_tr, cv=cv)

In [55]:
Q2_CV = round(r2_score(y_tr, y_pred_CV_svm), 2)
Q2_CV

0.53

In [56]:
RMSE_CV=float(round(np.sqrt(mean_squared_error(y_tr, y_pred_CV_svm)), 2))
RMSE_CV

0.58

#  Prediction for test set's molecules

In [ ]:
# y_ts = np.array(desc_ts, dtype=np.float32)
# y_ts = np.array(desc_ts, dtype=np.float32)

In [48]:
# x_ts = np.array(desc_ts, dtype=np.float32)
# y_ts = np.array(y_ts, dtype=np.float32)

In [71]:
y_pred_svm = best_svm.predict(desc_ts)

In [58]:
Q2_TS = round(r2_score(y_ts, y_pred_svm), 2)
Q2_TS

0.56

In [59]:
RMSE_TS=float(round(np.sqrt(mean_squared_error(y_ts, y_pred_svm)), 2))
RMSE_TS

0.57

save the model to disk

In [80]:
pickle.dump(best_svm, open('Models/MACCS/HDAC8_SVM_MACCS_FP.pkl', 'wb'))

load the model from disk

In [46]:
best_svm = pickle.load(open('Models/MACCS/HDAC8_SVM_MACCS_FP.pkl', 'rb'))

# 10. Y-randomization SVM model

In [ ]:
# permutations = 50
# score, permutation_scores, pvalue = permutation_test_score(best_svm, X_rfe, y_tr,
#                                                            cv=cv, scoring='r2',
#                                                            n_permutations=permutations,
#                                                            n_jobs=-1,
#                                                            verbose=1,
#                                                            random_state=seed)
# print('True score = ', score.round(3),
#       '\nY-randomization = ', np.mean(permutation_scores).round(2),
#       '\np-value = ', pvalue.round(4))

# Estimating applicability domain. Method - Euclidian distances, K=1

In [60]:
neighbors_k= pairwise_distances(desc_ws, n_jobs=-1)
neighbors_k.sort(0)

In [61]:
df_tr=pd.DataFrame(neighbors_k)
df_tr

,0,1,2,3,4,5,6,7,8,9,...,2200,2201,2202,2203,2204,2205,2206,2207,2208,2209
0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,3.316625,2.645751,1.000000,0.000000,2.449490,1.732051,1.414214,0.000000,2.828427,2.645751,...,1.000000,3.162278,3.605551,1.414214,3.162278,4.000000,0.000000,0.000000,1.000000,2.000000
2,3.316625,4.358899,2.645751,2.000000,2.449490,2.828427,1.414214,2.000000,3.000000,2.645751,...,2.236068,3.162278,3.605551,2.000000,3.162278,4.123106,1.414214,1.414214,1.414214,2.000000
3,3.605551,4.358899,3.316625,2.000000,2.828427,4.000000,2.000000,2.000000,3.162278,3.162278,...,2.828427,3.162278,3.605551,2.000000,3.316625,4.123106,1.414214,1.414214,2.000000,2.449490
4,3.605551,4.358899,3.316625,2.000000,3.162278,4.000000,2.000000,2.000000,3.316625,4.000000,...,3.605551,3.162278,3.741657,2.236068,3.605551,4.123106,1.414214,1.414214,2.000000,2.449490
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2205,8.366600,8.602325,7.937254,8.124038,8.124038,8.774964,8.246211,8.124038,8.000000,7.874008,...,8.426150,8.124038,8.306624,8.124038,8.426150,8.062258,7.937254,7.937254,8.944272,7.874008
2206,8.366600,8.602325,8.000000,8.185353,8.124038,8.774964,8.246211,8.185353,8.000000,7.874008,...,8.485281,8.124038,8.306624,8.124038,8.426150,8.124038,7.937254,7.937254,9.000000,7.874008
2207,8.426150,8.602325,8.000000,8.185353,8.124038,8.774964,8.306624,8.185353,8.000000,7.874008,...,8.485281,8.185353,8.306624,8.185353,8.485281,8.124038,8.000000,8.000000,9.000000,7.874008
2208,8.485281,8.602325,8.000000,8.246211,8.124038,8.831761,8.306624,8.246211,8.000000,7.874008,...,8.544004,8.185353,8.485281,8.185353,8.485281,8.124038,8.000000,8.000000,9.055385,7.874008


In [62]:
similarity= neighbors_k

In [63]:
Dmean=np.mean(similarity[1,:])

In [64]:
round(Dmean, 2)

np.float64(1.46)

In [65]:
std=np.std(similarity[1,:])

In [66]:
round(std, 2)

np.float64(1.21)

In [67]:
model_AD_limit=Dmean+std*0.5
print(np.round(model_AD_limit, 2))

2.07


In [72]:
neighbors_k_ts= pairwise_distances(desc_ws,Y=desc_ts, n_jobs=-1)
neighbors_k_ts.sort(0)

In [73]:
x_ts_AD=pd.DataFrame(neighbors_k_ts)
x_ts_AD

,0,1,2,3,4,5,6,7,8,9,...,543,544,545,546,547,548,549,550,551,552
0,1.414214,3.316625,2.000000,3.162278,1.000000,1.414214,2.828427,1.000000,2.828427,1.414214,...,3.000000,2.000000,1.000000,1.732051,1.414214,1.732051,2.000000,0.000000,1.732051,0.000000
1,2.828427,3.605551,2.449490,3.316625,1.000000,1.414214,2.828427,1.000000,2.828427,1.732051,...,3.162278,2.645751,2.000000,1.732051,1.414214,2.645751,2.000000,0.000000,2.645751,0.000000
2,3.000000,3.741657,2.645751,3.605551,2.236068,2.000000,3.000000,2.000000,2.828427,2.000000,...,3.316625,3.000000,2.449490,1.732051,1.732051,2.645751,2.236068,1.000000,2.645751,2.000000
3,3.162278,3.741657,2.828427,3.741657,2.236068,2.449490,3.605551,2.236068,2.828427,2.000000,...,3.316625,3.000000,2.645751,2.236068,1.732051,2.645751,2.236068,1.000000,2.645751,2.000000
4,3.316625,3.741657,3.000000,3.741657,2.236068,2.449490,3.605551,2.236068,2.828427,2.236068,...,3.464102,3.162278,2.645751,2.236068,1.732051,2.645751,2.236068,1.000000,2.828427,2.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2205,7.874008,7.810250,8.000000,8.000000,8.062258,8.124038,8.062258,8.062258,7.937254,8.124038,...,7.810250,8.124038,7.937254,7.937254,8.426150,7.810250,8.426150,8.306624,8.124038,8.602325
2206,7.874008,7.810250,8.000000,8.062258,8.124038,8.185353,8.124038,8.062258,7.937254,8.185353,...,7.810250,8.124038,7.937254,7.937254,8.426150,7.810250,8.426150,8.306624,8.185353,8.602325
2207,7.937254,7.874008,8.000000,8.124038,8.124038,8.185353,8.185353,8.062258,7.937254,8.246211,...,7.937254,8.185353,8.000000,7.937254,8.426150,7.810250,8.485281,8.306624,8.185353,8.602325
2208,8.000000,7.874008,8.000000,8.124038,8.185353,8.246211,8.246211,8.062258,8.000000,8.306624,...,8.000000,8.185353,8.000000,7.937254,8.426150,7.810250,8.485281,8.306624,8.185353,8.602325


In [74]:
similarity_ts= neighbors_k_ts
cpd_AD=similarity_ts[0,:]
cpd_value = np.round(cpd_AD, 3)
print(cpd_value)

[1.414 3.317 2.    3.162 1.    1.414 2.828 1.    2.828 1.414 4.359 1.
 0.    3.464 3.464 1.414 1.414 1.    0.    3.317 0.    0.    1.414 1.
 1.414 1.414 2.236 1.414 0.    2.    0.    3.    1.414 2.449 1.    2.828
 1.414 2.    3.464 3.464 4.    3.    2.646 0.    2.    1.414 1.    2.
 0.    2.236 1.414 0.    2.449 2.449 1.    1.414 2.646 3.606 0.    0.
 0.    2.    1.732 1.732 1.732 2.828 1.414 1.732 2.449 1.414 2.236 1.
 0.    2.    2.236 0.    3.742 0.    2.449 0.    1.    0.    0.    2.236
 2.    2.646 1.732 0.    0.    1.732 1.414 1.732 1.414 1.414 2.    2.449
 3.464 0.    0.    2.    0.    1.    0.    2.    1.732 1.732 0.    2.449
 1.    0.    0.    2.236 2.    2.236 0.    1.    1.414 2.236 0.    1.732
 2.    0.    0.    1.732 0.    1.    0.    0.    1.    1.414 2.646 3.317
 1.732 3.    0.    2.    1.414 2.646 0.    3.162 1.    1.414 0.    1.414
 1.    1.    0.    2.449 1.    1.    0.    0.    0.    2.236 2.646 1.
 4.472 2.236 2.    2.236 2.449 1.414 1.414 1.414 1.    1.414 0.    2.

In [75]:
cpd_AD = np.where(cpd_value <= model_AD_limit, True, False)
print(cpd_AD)

[ True False  True False  True  True False  True False  True False  True
  True False False  True  True  True  True False  True  True  True  True
  True  True False  True  True  True  True False  True False  True False
  True  True False False False False False  True  True  True  True  True
  True False  True  True False False  True  True False False  True  True
  True  True  True  True  True False  True  True False  True False  True
  True  True False  True False  True False  True  True  True  True False
  True False  True  True  True  True  True  True  True  True  True False
 False  True  True  True  True  True  True  True  True  True  True False
  True  True  True False  True False  True  True  True False  True  True
  True  True  True  True  True  True  True  True  True  True False False
  True False  True  True  True False  True False  True  True  True  True
  True  True  True False  True  True  True  True  True False False  True
 False False  True False False  True  True  True  T

In [76]:
print("Coverage = ", round(sum(cpd_AD) / len(cpd_AD), 2))

Coverage =  0.72


In [77]:
print("Indices of substances included in AD = ", np.where(cpd_AD != 0)[0])

Indices of substances included in AD =  [  0   2   4   5   7   9  11  12  15  16  17  18  20  21  22  23  24  25
  27  28  29  30  32  34  36  37  43  44  45  46  47  48  50  51  54  55
  58  59  60  61  62  63  64  66  67  69  71  72  73  75  77  79  80  81
  82  84  86  87  88  89  90  91  92  93  94  97  98  99 100 101 102 103
 104 105 106 108 109 110 112 114 115 116 118 119 120 121 122 123 124 125
 126 127 128 129 132 134 135 136 138 140 141 142 143 144 145 146 148 149
 150 151 152 155 158 161 162 163 164 165 166 168 169 170 171 172 173 174
 175 176 177 178 179 180 181 182 183 184 185 186 187 190 191 194 195 196
 199 200 202 205 206 207 208 209 211 213 214 217 218 219 220 221 222 224
 225 226 227 228 229 230 231 232 233 234 235 236 237 239 240 241 242 244
 245 246 247 248 249 250 251 252 256 258 259 261 262 263 264 265 266 267
 268 269 270 271 273 274 275 276 277 278 279 282 283 285 286 287 288 289
 290 293 294 295 296 298 299 301 304 305 308 310 311 312 313 315 316 319
 320 321 32

In [78]:
out_Ad=list(np.where(cpd_AD == 0)[0])

# Prediction only for molecules included in  AD

In [79]:
y_pred_svm_ad=list(y_pred_svm)

In [80]:
y_pred_svm_ad[:] = [x for i,x in enumerate(y_pred_svm_ad) if i not in out_Ad]

In [81]:
len(y_pred_svm_ad)

399

In [82]:
y_ts_ad=list(y_ts)

In [83]:
y_ts_ad[:] = [x for i,x in enumerate(y_ts_ad) if i not in out_Ad]

In [84]:
len(y_ts_ad)

399

In [85]:
Q2_TS = round(r2_score(y_ts_ad, y_pred_svm_ad), 2)
Q2_TS

0.64

In [86]:
RMSE_TS=float(round(np.sqrt(mean_squared_error(y_ts_ad, y_pred_svm_ad)), 2))
RMSE_TS

0.51

# Multi-layer Perceptron regressor

In [106]:
from sklearn.neural_network import MLPRegressor

In [107]:
param_grid ={"hidden_layer_sizes": [(400, 300, 200, 100),(100, 100, 100), (10, 10, 10),(50,)], "activation": ["tanh", "relu"], "solver": ["lbfgs", "sgd", "adam"], "alpha": [0.00005,0.0005], 'max_iter': [1000, 2000]}

In [108]:
m = GridSearchCV(MLPRegressor(), param_grid, n_jobs=-1, cv=cv, verbose=1)

In [109]:
m.fit(desc_ws, y_tr)

Fitting 5 folds for each of 96 candidates, totalling 480 fits


,estimator,MLPRegressor()
,param_grid,"{'activation': ['tanh', 'relu'], 'alpha': [5e-05, 0.0005], 'hidden_layer_sizes': [(400, ...), (100, ...), ...], 'max_iter': [1000, 2000], ...}"
,scoring,None
,n_jobs,-1
,refit,True
,cv,KFold(n_split... shuffle=True)
,verbose,1
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,loss,'squared_error'


In [110]:
best_MLPR = m.best_estimator_

In [88]:
y_pred_ws_best_MLPR = best_MLPR.predict(desc_ws)

In [94]:
R2_WS = round(r2_score(y_tr, y_pred_ws_best_MLPR), 2)
R2_WS

0.89

In [95]:
RMSE_WS=float(round(np.sqrt(mean_squared_error(y_tr, y_pred_ws_best_MLPR)), 2))
RMSE_WS

0.28

In [96]:
y_pred_CV_MLPR = cross_val_predict(best_MLPR, desc_ws, y_tr, cv=cv)

In [97]:
Q2_CV = round(r2_score(y_tr, y_pred_CV_MLPR), 2)
Q2_CV

0.41

In [98]:
RMSE_CV=float(round(np.sqrt(mean_squared_error(y_tr, y_pred_CV_MLPR)), 2))
RMSE_CV

0.65

# Prediction for test set's molecules

In [99]:
y_pred_MLPR = best_MLPR.predict(desc_ts)

In [100]:
Q2_TS = round(r2_score(y_ts, y_pred_MLPR), 2)
Q2_TS

0.48

In [102]:
RMSE_TS=float(round(np.sqrt(mean_squared_error(y_ts, y_pred_MLPR)), 2))
RMSE_TS

0.61

save the model to disk

In [141]:
pickle.dump(best_MLPR, open('models/MACCS/MLPR_MACCS.pkl', 'wb'))

load the model from disk

In [87]:
best_MLPR = pickle.load(open('models/MACCS/MLPR_MACCS.pkl', 'rb'))

#  Estimating applicability domain. Method - Euclidian distances, K=1

In [103]:
neighbors_k= pairwise_distances(desc_ws, n_jobs=-1)
neighbors_k.sort(0)

In [104]:
df_tr=pd.DataFrame(neighbors_k)
df_tr

,0,1,2,3,4,5,6,7,8,9,...,2200,2201,2202,2203,2204,2205,2206,2207,2208,2209
0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,3.316625,2.645751,1.000000,0.000000,2.449490,1.732051,1.414214,0.000000,2.828427,2.645751,...,1.000000,3.162278,3.605551,1.414214,3.162278,4.000000,0.000000,0.000000,1.000000,2.000000
2,3.316625,4.358899,2.645751,2.000000,2.449490,2.828427,1.414214,2.000000,3.000000,2.645751,...,2.236068,3.162278,3.605551,2.000000,3.162278,4.123106,1.414214,1.414214,1.414214,2.000000
3,3.605551,4.358899,3.316625,2.000000,2.828427,4.000000,2.000000,2.000000,3.162278,3.162278,...,2.828427,3.162278,3.605551,2.000000,3.316625,4.123106,1.414214,1.414214,2.000000,2.449490
4,3.605551,4.358899,3.316625,2.000000,3.162278,4.000000,2.000000,2.000000,3.316625,4.000000,...,3.605551,3.162278,3.741657,2.236068,3.605551,4.123106,1.414214,1.414214,2.000000,2.449490
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2205,8.366600,8.602325,7.937254,8.124038,8.124038,8.774964,8.246211,8.124038,8.000000,7.874008,...,8.426150,8.124038,8.306624,8.124038,8.426150,8.062258,7.937254,7.937254,8.944272,7.874008
2206,8.366600,8.602325,8.000000,8.185353,8.124038,8.774964,8.246211,8.185353,8.000000,7.874008,...,8.485281,8.124038,8.306624,8.124038,8.426150,8.124038,7.937254,7.937254,9.000000,7.874008
2207,8.426150,8.602325,8.000000,8.185353,8.124038,8.774964,8.306624,8.185353,8.000000,7.874008,...,8.485281,8.185353,8.306624,8.185353,8.485281,8.124038,8.000000,8.000000,9.000000,7.874008
2208,8.485281,8.602325,8.000000,8.246211,8.124038,8.831761,8.306624,8.246211,8.000000,7.874008,...,8.544004,8.185353,8.485281,8.185353,8.485281,8.124038,8.000000,8.000000,9.055385,7.874008


In [105]:
similarity= neighbors_k

In [106]:
Dmean=np.mean(similarity[1,:])

In [107]:
round(Dmean, 2)

np.float64(1.46)

In [108]:
std=np.std(similarity[1,:])

In [109]:
round(std, 2)

np.float64(1.21)

In [110]:
model_AD_limit=Dmean+std*0.5
print(np.round(model_AD_limit, 2))

2.07


In [111]:
neighbors_k_ts= pairwise_distances(desc_ws,Y=desc_ts, n_jobs=-1)
neighbors_k_ts.sort(0)

In [112]:
x_ts_AD=pd.DataFrame(neighbors_k_ts)
x_ts_AD

,0,1,2,3,4,5,6,7,8,9,...,543,544,545,546,547,548,549,550,551,552
0,1.414214,3.316625,2.000000,3.162278,1.000000,1.414214,2.828427,1.000000,2.828427,1.414214,...,3.000000,2.000000,1.000000,1.732051,1.414214,1.732051,2.000000,0.000000,1.732051,0.000000
1,2.828427,3.605551,2.449490,3.316625,1.000000,1.414214,2.828427,1.000000,2.828427,1.732051,...,3.162278,2.645751,2.000000,1.732051,1.414214,2.645751,2.000000,0.000000,2.645751,0.000000
2,3.000000,3.741657,2.645751,3.605551,2.236068,2.000000,3.000000,2.000000,2.828427,2.000000,...,3.316625,3.000000,2.449490,1.732051,1.732051,2.645751,2.236068,1.000000,2.645751,2.000000
3,3.162278,3.741657,2.828427,3.741657,2.236068,2.449490,3.605551,2.236068,2.828427,2.000000,...,3.316625,3.000000,2.645751,2.236068,1.732051,2.645751,2.236068,1.000000,2.645751,2.000000
4,3.316625,3.741657,3.000000,3.741657,2.236068,2.449490,3.605551,2.236068,2.828427,2.236068,...,3.464102,3.162278,2.645751,2.236068,1.732051,2.645751,2.236068,1.000000,2.828427,2.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2205,7.874008,7.810250,8.000000,8.000000,8.062258,8.124038,8.062258,8.062258,7.937254,8.124038,...,7.810250,8.124038,7.937254,7.937254,8.426150,7.810250,8.426150,8.306624,8.124038,8.602325
2206,7.874008,7.810250,8.000000,8.062258,8.124038,8.185353,8.124038,8.062258,7.937254,8.185353,...,7.810250,8.124038,7.937254,7.937254,8.426150,7.810250,8.426150,8.306624,8.185353,8.602325
2207,7.937254,7.874008,8.000000,8.124038,8.124038,8.185353,8.185353,8.062258,7.937254,8.246211,...,7.937254,8.185353,8.000000,7.937254,8.426150,7.810250,8.485281,8.306624,8.185353,8.602325
2208,8.000000,7.874008,8.000000,8.124038,8.185353,8.246211,8.246211,8.062258,8.000000,8.306624,...,8.000000,8.185353,8.000000,7.937254,8.426150,7.810250,8.485281,8.306624,8.185353,8.602325


In [113]:
similarity_ts= neighbors_k_ts
cpd_AD=similarity_ts[0,:]
cpd_value = np.round(cpd_AD, 3)
print(cpd_value)

[1.414 3.317 2.    3.162 1.    1.414 2.828 1.    2.828 1.414 4.359 1.
 0.    3.464 3.464 1.414 1.414 1.    0.    3.317 0.    0.    1.414 1.
 1.414 1.414 2.236 1.414 0.    2.    0.    3.    1.414 2.449 1.    2.828
 1.414 2.    3.464 3.464 4.    3.    2.646 0.    2.    1.414 1.    2.
 0.    2.236 1.414 0.    2.449 2.449 1.    1.414 2.646 3.606 0.    0.
 0.    2.    1.732 1.732 1.732 2.828 1.414 1.732 2.449 1.414 2.236 1.
 0.    2.    2.236 0.    3.742 0.    2.449 0.    1.    0.    0.    2.236
 2.    2.646 1.732 0.    0.    1.732 1.414 1.732 1.414 1.414 2.    2.449
 3.464 0.    0.    2.    0.    1.    0.    2.    1.732 1.732 0.    2.449
 1.    0.    0.    2.236 2.    2.236 0.    1.    1.414 2.236 0.    1.732
 2.    0.    0.    1.732 0.    1.    0.    0.    1.    1.414 2.646 3.317
 1.732 3.    0.    2.    1.414 2.646 0.    3.162 1.    1.414 0.    1.414
 1.    1.    0.    2.449 1.    1.    0.    0.    0.    2.236 2.646 1.
 4.472 2.236 2.    2.236 2.449 1.414 1.414 1.414 1.    1.414 0.    2.

In [114]:
cpd_AD = np.where(cpd_value <= model_AD_limit, True, False)
print(cpd_AD)

[ True False  True False  True  True False  True False  True False  True
  True False False  True  True  True  True False  True  True  True  True
  True  True False  True  True  True  True False  True False  True False
  True  True False False False False False  True  True  True  True  True
  True False  True  True False False  True  True False False  True  True
  True  True  True  True  True False  True  True False  True False  True
  True  True False  True False  True False  True  True  True  True False
  True False  True  True  True  True  True  True  True  True  True False
 False  True  True  True  True  True  True  True  True  True  True False
  True  True  True False  True False  True  True  True False  True  True
  True  True  True  True  True  True  True  True  True  True False False
  True False  True  True  True False  True False  True  True  True  True
  True  True  True False  True  True  True  True  True False False  True
 False False  True False False  True  True  True  T

In [115]:
print("Coverage = ", round(sum(cpd_AD) / len(cpd_AD), 2))

Coverage =  0.72


In [116]:
print("Indices of substances included in AD = ", np.where(cpd_AD != 0)[0])

Indices of substances included in AD =  [  0   2   4   5   7   9  11  12  15  16  17  18  20  21  22  23  24  25
  27  28  29  30  32  34  36  37  43  44  45  46  47  48  50  51  54  55
  58  59  60  61  62  63  64  66  67  69  71  72  73  75  77  79  80  81
  82  84  86  87  88  89  90  91  92  93  94  97  98  99 100 101 102 103
 104 105 106 108 109 110 112 114 115 116 118 119 120 121 122 123 124 125
 126 127 128 129 132 134 135 136 138 140 141 142 143 144 145 146 148 149
 150 151 152 155 158 161 162 163 164 165 166 168 169 170 171 172 173 174
 175 176 177 178 179 180 181 182 183 184 185 186 187 190 191 194 195 196
 199 200 202 205 206 207 208 209 211 213 214 217 218 219 220 221 222 224
 225 226 227 228 229 230 231 232 233 234 235 236 237 239 240 241 242 244
 245 246 247 248 249 250 251 252 256 258 259 261 262 263 264 265 266 267
 268 269 270 271 273 274 275 276 277 278 279 282 283 285 286 287 288 289
 290 293 294 295 296 298 299 301 304 305 308 310 311 312 313 315 316 319
 320 321 32

In [117]:
out_Ad=list(np.where(cpd_AD == 0)[0])

# Prediction only for molecules included in  AD

In [118]:
y_pred_MLPR_ad=list(y_pred_MLPR)

In [119]:
y_pred_MLPR_ad[:] = [x for i,x in enumerate(y_pred_MLPR_ad) if i not in out_Ad]

In [120]:
len(y_pred_MLPR_ad)

399

In [121]:
y_ts_ad=list(y_ts)

In [122]:
y_ts_ad[:] = [x for i,x in enumerate(y_ts_ad) if i not in out_Ad]

In [123]:
len(y_ts_ad)

399

In [124]:
Q2_TS = round(r2_score(y_ts_ad, y_pred_MLPR_ad), 2)
Q2_TS

0.62

In [125]:
RMSE_TS=float(round(np.sqrt(mean_squared_error(y_ts_ad, y_pred_MLPR_ad)), 2))
RMSE_TS

0.53

# k-nearest neighbors

In [129]:
k_range = list(range(1, 31))
param_grid = dict(n_neighbors=k_range)

In [130]:
m = GridSearchCV(KNeighborsRegressor(), param_grid, n_jobs=-1, cv=cv, verbose=1)

In [131]:
m.fit(desc_ws, y_tr)

Fitting 5 folds for each of 30 candidates, totalling 150 fits


,estimator,KNeighborsRegressor()
,param_grid,"{'n_neighbors': [1, 2, ...]}"
,scoring,None
,n_jobs,-1
,refit,True
,cv,KFold(n_split... shuffle=True)
,verbose,1
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,n_neighbors,4


In [132]:
best_kNN = m.best_estimator_

In [133]:
m.best_params_

{'n_neighbors': 4}

In [134]:
y_pred_ws_kNN = best_kNN.predict(desc_ws)

In [135]:
R2_WS = round(r2_score(y_tr, y_pred_ws_kNN), 2)
R2_WS

0.71

In [136]:
RMSE_WS=float(round(np.sqrt(mean_squared_error(y_tr, y_pred_ws_kNN)), 2))
RMSE_WS

0.46

In [137]:
y_pred_CV_kNN = cross_val_predict(best_kNN, desc_ws, y_tr, cv=cv)

In [138]:
y_pred_CV_kNN

array([6.585    , 5.45     , 4.615    , ..., 7.855    , 6.4425   ,
       7.4925003], shape=(2210,), dtype=float32)

In [139]:
Q2_CV = round(r2_score(y_tr, y_pred_CV_kNN), 2)
Q2_CV

0.45

In [140]:
RMSE_CV=float(round(np.sqrt(mean_squared_error(y_tr, y_pred_CV_kNN)), 2))
RMSE_CV

0.63

# 9. Prediction for test set's molecules

In [141]:
x_ts = np.array(desc_ws, dtype=np.float32)
y_ts = np.array(y_ts, dtype=np.float32)

In [142]:
y_pred_kNN = best_kNN.predict(desc_ts)

In [143]:
Q2_TS = round(r2_score(y_ts, y_pred_kNN), 2)
Q2_TS

0.52

In [144]:
RMSE_TS=float(round(np.sqrt(mean_squared_error(y_ts, y_pred_kNN)), 2))
RMSE_TS

0.59

# save the model to disk

In [145]:
pickle.dump(best_kNN, open('models/MACCS/MACCS_kNN.pkl', 'wb'))

# load the model from disk

In [146]:
best_kNN = pickle.load(open('models/MACCS/MACCS_kNN.pkl', 'rb'))

#  Estimating applicability domain. Method - Euclidian distances, K=1

In [147]:
neighbors_k= pairwise_distances(desc_ws, n_jobs=-1)
neighbors_k.sort(0)

In [148]:
df_tr=pd.DataFrame(neighbors_k)
df_tr

,0,1,2,3,4,5,6,7,8,9,...,2200,2201,2202,2203,2204,2205,2206,2207,2208,2209
0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,3.316625,2.645751,1.000000,0.000000,2.449490,1.732051,1.414214,0.000000,2.828427,2.645751,...,1.000000,3.162278,3.605551,1.414214,3.162278,4.000000,0.000000,0.000000,1.000000,2.000000
2,3.316625,4.358899,2.645751,2.000000,2.449490,2.828427,1.414214,2.000000,3.000000,2.645751,...,2.236068,3.162278,3.605551,2.000000,3.162278,4.123106,1.414214,1.414214,1.414214,2.000000
3,3.605551,4.358899,3.316625,2.000000,2.828427,4.000000,2.000000,2.000000,3.162278,3.162278,...,2.828427,3.162278,3.605551,2.000000,3.316625,4.123106,1.414214,1.414214,2.000000,2.449490
4,3.605551,4.358899,3.316625,2.000000,3.162278,4.000000,2.000000,2.000000,3.316625,4.000000,...,3.605551,3.162278,3.741657,2.236068,3.605551,4.123106,1.414214,1.414214,2.000000,2.449490
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2205,8.366600,8.602325,7.937254,8.124038,8.124038,8.774964,8.246211,8.124038,8.000000,7.874008,...,8.426150,8.124038,8.306624,8.124038,8.426150,8.062258,7.937254,7.937254,8.944272,7.874008
2206,8.366600,8.602325,8.000000,8.185353,8.124038,8.774964,8.246211,8.185353,8.000000,7.874008,...,8.485281,8.124038,8.306624,8.124038,8.426150,8.124038,7.937254,7.937254,9.000000,7.874008
2207,8.426150,8.602325,8.000000,8.185353,8.124038,8.774964,8.306624,8.185353,8.000000,7.874008,...,8.485281,8.185353,8.306624,8.185353,8.485281,8.124038,8.000000,8.000000,9.000000,7.874008
2208,8.485281,8.602325,8.000000,8.246211,8.124038,8.831761,8.306624,8.246211,8.000000,7.874008,...,8.544004,8.185353,8.485281,8.185353,8.485281,8.124038,8.000000,8.000000,9.055385,7.874008


In [149]:
similarity= neighbors_k

In [150]:
Dmean=np.mean(similarity[1,:])

In [151]:
round(Dmean, 2)

np.float64(1.46)

In [152]:
std=np.std(similarity[1,:])

In [153]:
round(std, 2)

np.float64(1.21)

In [154]:
model_AD_limit=Dmean+std*0.5
print(np.round(model_AD_limit, 2))

2.07


In [155]:
neighbors_k_ts= pairwise_distances(desc_ws,Y=desc_ts, n_jobs=-1)
neighbors_k_ts.sort(0)

In [156]:
x_ts_AD=pd.DataFrame(neighbors_k_ts)
x_ts_AD

,0,1,2,3,4,5,6,7,8,9,...,543,544,545,546,547,548,549,550,551,552
0,1.414214,3.316625,2.000000,3.162278,1.000000,1.414214,2.828427,1.000000,2.828427,1.414214,...,3.000000,2.000000,1.000000,1.732051,1.414214,1.732051,2.000000,0.000000,1.732051,0.000000
1,2.828427,3.605551,2.449490,3.316625,1.000000,1.414214,2.828427,1.000000,2.828427,1.732051,...,3.162278,2.645751,2.000000,1.732051,1.414214,2.645751,2.000000,0.000000,2.645751,0.000000
2,3.000000,3.741657,2.645751,3.605551,2.236068,2.000000,3.000000,2.000000,2.828427,2.000000,...,3.316625,3.000000,2.449490,1.732051,1.732051,2.645751,2.236068,1.000000,2.645751,2.000000
3,3.162278,3.741657,2.828427,3.741657,2.236068,2.449490,3.605551,2.236068,2.828427,2.000000,...,3.316625,3.000000,2.645751,2.236068,1.732051,2.645751,2.236068,1.000000,2.645751,2.000000
4,3.316625,3.741657,3.000000,3.741657,2.236068,2.449490,3.605551,2.236068,2.828427,2.236068,...,3.464102,3.162278,2.645751,2.236068,1.732051,2.645751,2.236068,1.000000,2.828427,2.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2205,7.874008,7.810250,8.000000,8.000000,8.062258,8.124038,8.062258,8.062258,7.937254,8.124038,...,7.810250,8.124038,7.937254,7.937254,8.426150,7.810250,8.426150,8.306624,8.124038,8.602325
2206,7.874008,7.810250,8.000000,8.062258,8.124038,8.185353,8.124038,8.062258,7.937254,8.185353,...,7.810250,8.124038,7.937254,7.937254,8.426150,7.810250,8.426150,8.306624,8.185353,8.602325
2207,7.937254,7.874008,8.000000,8.124038,8.124038,8.185353,8.185353,8.062258,7.937254,8.246211,...,7.937254,8.185353,8.000000,7.937254,8.426150,7.810250,8.485281,8.306624,8.185353,8.602325
2208,8.000000,7.874008,8.000000,8.124038,8.185353,8.246211,8.246211,8.062258,8.000000,8.306624,...,8.000000,8.185353,8.000000,7.937254,8.426150,7.810250,8.485281,8.306624,8.185353,8.602325


In [157]:
similarity_ts= neighbors_k_ts
cpd_AD=similarity_ts[0,:]
cpd_value = np.round(cpd_AD, 3)
print(cpd_value)

[1.414 3.317 2.    3.162 1.    1.414 2.828 1.    2.828 1.414 4.359 1.
 0.    3.464 3.464 1.414 1.414 1.    0.    3.317 0.    0.    1.414 1.
 1.414 1.414 2.236 1.414 0.    2.    0.    3.    1.414 2.449 1.    2.828
 1.414 2.    3.464 3.464 4.    3.    2.646 0.    2.    1.414 1.    2.
 0.    2.236 1.414 0.    2.449 2.449 1.    1.414 2.646 3.606 0.    0.
 0.    2.    1.732 1.732 1.732 2.828 1.414 1.732 2.449 1.414 2.236 1.
 0.    2.    2.236 0.    3.742 0.    2.449 0.    1.    0.    0.    2.236
 2.    2.646 1.732 0.    0.    1.732 1.414 1.732 1.414 1.414 2.    2.449
 3.464 0.    0.    2.    0.    1.    0.    2.    1.732 1.732 0.    2.449
 1.    0.    0.    2.236 2.    2.236 0.    1.    1.414 2.236 0.    1.732
 2.    0.    0.    1.732 0.    1.    0.    0.    1.    1.414 2.646 3.317
 1.732 3.    0.    2.    1.414 2.646 0.    3.162 1.    1.414 0.    1.414
 1.    1.    0.    2.449 1.    1.    0.    0.    0.    2.236 2.646 1.
 4.472 2.236 2.    2.236 2.449 1.414 1.414 1.414 1.    1.414 0.    2.

In [158]:
cpd_AD = np.where(cpd_value <= model_AD_limit, True, False)
print(cpd_AD)

[ True False  True False  True  True False  True False  True False  True
  True False False  True  True  True  True False  True  True  True  True
  True  True False  True  True  True  True False  True False  True False
  True  True False False False False False  True  True  True  True  True
  True False  True  True False False  True  True False False  True  True
  True  True  True  True  True False  True  True False  True False  True
  True  True False  True False  True False  True  True  True  True False
  True False  True  True  True  True  True  True  True  True  True False
 False  True  True  True  True  True  True  True  True  True  True False
  True  True  True False  True False  True  True  True False  True  True
  True  True  True  True  True  True  True  True  True  True False False
  True False  True  True  True False  True False  True  True  True  True
  True  True  True False  True  True  True  True  True False False  True
 False False  True False False  True  True  True  T

In [159]:
print("Coverage = ", round(sum(cpd_AD) / len(cpd_AD), 2))

Coverage =  0.72


In [160]:
print("Indices of substances included in AD = ", np.where(cpd_AD != 0)[0])

Indices of substances included in AD =  [  0   2   4   5   7   9  11  12  15  16  17  18  20  21  22  23  24  25
  27  28  29  30  32  34  36  37  43  44  45  46  47  48  50  51  54  55
  58  59  60  61  62  63  64  66  67  69  71  72  73  75  77  79  80  81
  82  84  86  87  88  89  90  91  92  93  94  97  98  99 100 101 102 103
 104 105 106 108 109 110 112 114 115 116 118 119 120 121 122 123 124 125
 126 127 128 129 132 134 135 136 138 140 141 142 143 144 145 146 148 149
 150 151 152 155 158 161 162 163 164 165 166 168 169 170 171 172 173 174
 175 176 177 178 179 180 181 182 183 184 185 186 187 190 191 194 195 196
 199 200 202 205 206 207 208 209 211 213 214 217 218 219 220 221 222 224
 225 226 227 228 229 230 231 232 233 234 235 236 237 239 240 241 242 244
 245 246 247 248 249 250 251 252 256 258 259 261 262 263 264 265 266 267
 268 269 270 271 273 274 275 276 277 278 279 282 283 285 286 287 288 289
 290 293 294 295 296 298 299 301 304 305 308 310 311 312 313 315 316 319
 320 321 32

In [161]:
out_Ad=list(np.where(cpd_AD == 0)[0])

# Prediction only for molecules included in  AD

In [164]:
y_pred_kNN_ad=list(y_pred_kNN)

In [165]:
y_pred_kNN_ad[:] = [x for i,x in enumerate(y_pred_kNN_ad) if i not in out_Ad]

In [166]:
len(y_pred_kNN_ad)

399

In [167]:
y_ts_ad=list(y_ts)

In [168]:
y_ts_ad[:] = [x for i,x in enumerate(y_ts_ad) if i not in out_Ad]

In [169]:
len(y_ts_ad)

399

In [170]:
Q2_TS = round(r2_score(y_ts_ad, y_pred_kNN_ad), 2)
Q2_TS

0.61

In [171]:
RMSE_TS=float(round(np.sqrt(mean_squared_error(y_ts_ad, y_pred_kNN_ad)), 2))
RMSE_TS

0.53